# 採点役だけの確認（docs/v2-light-model-plan.md §4 Step 1・Colab T4 用）

10/22 の Step 1 で採点役 `stelterlab/phi-4-AWQ` が、正解の分かっている 5 例に **全部 1.0** を付けた（確認は fail）。
原因を見るため、採点役だけを起動して 5 例の **生の返答** を出す。索引・候補モデルは使わない（GPU 10〜15 分）。

- 採点の文面・設定は本番（`evaluation/judges.py` の `judge`）と同じ。文面は変えない
- 送る文はテンプレートを当てた後の形でも出す（チャットの書式が効いているかを見る）
- 全部セルの出力に出す（ランタイムが切れてもファイルを取りに行かなくてよいように）

読み方:
- 根拠コメントが誤り（本文に無い承認者・逆の結論）を指摘しているのに 1.0 → 採点役が甘い。採点役を替える
- 根拠コメントが回答を見ていない・一般論 → 入力か書式の問題。直して確かめ直す


In [ ]:
# 準備（vLLM だけ。RAG 側の仮想環境は使わない）
!pip -q install uv
!git clone -q --depth 1 -b redesign/v2 https://github.com/Maaa2005/local-rag-agent.git /content/repo
!uv venv -q /content/vllm-env && uv pip install -q --python /content/vllm-env/bin/python vllm


In [ ]:
import json, os, pathlib, re, signal, subprocess, sys, time, urllib.request
REPO = "/content/repo"
sys.path.insert(0, REPO)
from evaluation import judges                      # 標準ライブラリだけ
from evaluation.colab import step05 as s5

JUDGE_MODEL = "stelterlab/phi-4-AWQ"
JUDGE_LEN, JUDGE_UTIL = 8192, 0.85                 # Step 1 と同じ
PORT = 8001
BASE = f"http://127.0.0.1:{PORT}"
MAX_TOKENS, TEMPERATURE = 1024, 0.1                # backend/app/config.py の既定（本番の採点と同じ）
LOG = "/content/vllm_judge_probe.log"
REV = s5.hf_revision(JUDGE_MODEL)
print("revision:", REV)

def post(path, body, timeout=600):
    req = urllib.request.Request(BASE + path, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())

argv = s5.vllm_argv("/content/vllm-env/bin/vllm", JUDGE_MODEL, PORT, JUDGE_LEN, JUDGE_UTIL, REV.get("sha"))
print(" ".join(argv))
proc = subprocess.Popen(argv, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)
t = time.time(); ready = False
while time.time() - t < 1800 and proc.poll() is None:
    try:
        urllib.request.urlopen(BASE + "/v1/models", timeout=3); ready = True; break
    except Exception:
        time.sleep(3)
print("ready:", ready, f"{time.time() - t:.0f} 秒")
if not ready:
    print("\n".join(pathlib.Path(LOG).read_text(errors="replace").splitlines()[-40:]))
    raise RuntimeError("採点役が起動しない")


In [ ]:
# 起動ログから、型・量子化・チャットの書式に関する行を抜き出す
lines = pathlib.Path(LOG).read_text(errors="replace").splitlines()
pat = re.compile(r"dtype|awq|quant|chat.?template|template|WARNING|tokenizer|max_model_len|bfloat16|float16", re.I)
for l in [l for l in lines if pat.search(l)][:60]:
    print(l[:300])


In [ ]:
# 5 例を本番と同じ文面・設定で送り、生の返答を出す
rows = [json.loads(l) for l in pathlib.Path(REPO, "evaluation/v2/judge_precheck.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
SYSTEM = "あなたは公平な RAG 評価者です。必ず JSON のみで答えてください。"   # judges.judge と同じ
PROBE = {"model": JUDGE_MODEL, "revision": REV.get("sha"), "max_tokens": MAX_TOKENS, "temperature": TEMPERATURE, "rows": []}
for i, r in enumerate(rows):
    prompt = judges._PROMPT.format(
        question=r["question"], chunks=judges._format_chunks(r["chunks"]),
        answer=r["answer"] or "(空)",
        keywords=", ".join(r["expected_keywords"]) if r["expected_keywords"] else "(なし)")
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
    if i == 0:
        # テンプレートを当てた後の文（モデルに実際に入る形）。書式の目印が入っているかを見る
        try:
            tk = post("/tokenize", {"model": "local-llm", "messages": messages, "add_generation_prompt": True})
            dt = post("/detokenize", {"model": "local-llm", "tokens": tk["tokens"]})
            text = dt.get("prompt", "")
            print("=== テンプレート後の文", tk.get("count"), "トークン ===")
            print(text[:400]); print("…"); print(text[-300:])
        except Exception as e:
            print("tokenize/detokenize 失敗:", type(e).__name__, e)
    t0 = time.time()
    res = post("/v1/chat/completions", {"model": "local-llm", "messages": messages,
                                        "max_tokens": MAX_TOKENS, "temperature": TEMPERATURE})
    text = res["choices"][0]["message"]["content"] or ""
    parsed = judges._extract_json(text)
    rec = {"kind": r["kind"], "id": r["id"], "answer": r["answer"], "raw_text": text, "parsed": parsed,
           "finish_reason": res["choices"][0].get("finish_reason"), "usage": res.get("usage"),
           "seconds": round(time.time() - t0, 2)}
    PROBE["rows"].append(rec)
    print(f"\n=== {r['kind']} ({r['id']}) {rec['seconds']} 秒 finish={rec['finish_reason']} usage={rec['usage']} ===")
    print("回答:", r["answer"])
    print("--- 生の返答 ---"); print(text)
    print("--- 取り出した JSON ---"); print(json.dumps(parsed, ensure_ascii=False))


In [ ]:
# まとめ（全部セルの出力に残す）→ 採点役を落とす
print(json.dumps({"summary": [{k: x[k] for k in ("kind", "finish_reason", "seconds")} |
                              {m: x["parsed"].get(m) for m in ("faithfulness", "answer_relevance", "context_relevance")}
                              for x in PROBE["rows"]]}, ensure_ascii=False, indent=1))
os.killpg(proc.pid, signal.SIGKILL); proc.wait()
print("採点役を落とした")
